In [3]:
!pip install -q -U google-genai pypdf

In [4]:
from google import genai
from google.colab import userdata
from google.genai import types
import numpy as np

client=genai.Client(api_key=userdata.get("GEMINI_API_KEYS"))
EMB_MODEL="gemini-embedding-001"
EMB_DIM=768
MODEL="gemini-3.5-flash-lite"

In [5]:
#1Prepare Document
#1.1 Upload the document
from google.colab import files
uploaded=files.upload()
pdf_name=list(uploaded.keys())[0]
print("PDF UPLOADED: ",pdf_name)

Saving College_FAQ_Knowledge_Base.pdf to College_FAQ_Knowledge_Base.pdf
PDF UPLOADED:  College_FAQ_Knowledge_Base.pdf


In [8]:
#1.2 Extract text from PDF Document
from pypdf import PdfReader
reader=PdfReader(pdf_name)
print("NUMBER OF PAGES: ",len(reader.pages))
text=""
for page in reader.pages:
  text+=page.extract_text()+"\n"

print(f"NUMBER OF CHARACTERS:{len(text)}")

NUMBER OF PAGES:  3
NUMBER OF CHARACTERS:5745


In [9]:
#1.3 Chunking (overalapping chunks)
def chunk_text(text,chunk_size=800,overlap=15):
  start=0
  chunks=[]
  while start<len(text):
    end=start+chunk_size
    chunks.append(text[start:end])
    start=end-overlap
  return chunks

chunks=chunk_text(text)
print(f"NUMBER OF CHUNKS:{len(chunks)}")

print("CHUNKING DONE")

NUMBER OF CHUNKS:8
CHUNKING DONE


In [12]:
#1.4 embed each chunk
def embed_chunk(chunk):
  response=client.models.embed_content(
      model=EMB_MODEL,
      contents=chunk,
      config=types.EmbedContentConfig(
          output_dimensionality=EMB_DIM
      )
  )
  return response.embeddings[0].values
chunk_embeddings=[]
for chunk in chunks:
  chunk_embeddings.append(embed_chunk(chunk))
print("EMBEDDING DONE")
chunk_embeddings=np.array(chunk_embeddings)
print(f"SHAPE:{chunk_embeddings.shape}")

EMBEDDING DONE
SHAPE:(8, 768)


In [13]:
#Finding closest question through cosine similarity
def cosine_sim(a,b):
  a=np.array(a)
  b=np.array(b)

  return float((np.dot(a,b))/(np.linalg.norm(a)*(np.linalg.norm(b))))

In [14]:
#1.4 retrieving chunk relavent to question
def retrieve(question,k=3):
  question_embed=embed_chunk(question)
  similarities=[]
  for i,chunk_embed in enumerate(chunk_embeddings):
    score=cosine_sim(question_embed,chunk_embed)
    similarities.append((i,score))
  similarities.sort(key=lambda x:x[1],reverse=True)

  similarities=similarities[:k]
  print("FOUND RELAVENT CHUNK")
  return similarities


In [17]:
#2Augumentation
system_instruction="""
                  Act as AI College Assistant who has frequently asked knowledge database, when session starts reply HI I AM COLLAGE FAQ ASSISTANT .
                  Answer to questions asked by user,if the information is not provided then dont provide with assumed/irreleavent data.
                  Be polite,if any unethical or foul language is used request to not use foul languages.
                  Also respond by data provided by user in the history of perticular session
                   """

#chatbot creation
chats=client.chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0.6,
        max_output_tokens=1500
    ),
    history=[]
)

print("NOTE:ENTER EXIT,BYE,QUIT to end session")

while True:
  user_input=input("AI:Enter the query: ")
  if user_input in ['exit','bye','quit']:
    print("EXITING.......\nEXITED...THANK YOU")
    break
  similarities=retrieve(user_input)
  print("RETRIEVED CHUNKS")
  sim_chunks=[]
  for index,score in similarities:
    sim_chunks.append(chunks[index])
  prompt=f"""
        So answer user question={user_input} by using pdf data={sim_chunks} where similar chunks is found relavent to question.
        Provide answer in professional structure.NOTE ACT AS COLLEGE FAQ ASSISTANT.
        """
  response=chats.send_message(prompt)
  print(f"AI:{response.text}")

NOTE:ENTER EXIT,BYE,QUIT to end session
AI:Enter the query: hi i am kushal
FOUND RELAVENT CHUNK
RETRIEVED CHUNKS
AI:HI I AM COLLAGE FAQ ASSISTANT.

Hello Kushal! How can I assist you with your college-related questions today?
AI:Enter the query: creiteria for engineering
FOUND RELAVENT CHUNK
RETRIEVED CHUNKS
AI:Hello Kushal. 

Based on the institutional guidelines provided in our knowledge base, here are the eligibility requirements for B.E. admission:

* **Educational Qualification:** Students must have completed 10+2 or an equivalent qualification.
* **Compulsory Subjects:** Physics and Mathematics must be taken as compulsory subjects.
* **Additional Requirements:** Candidates must also meet the institution's minimum admission requirements.

Please let me know if you have any other questions regarding admissions or other college procedures!
AI:Enter the query: where is bmsit
FOUND RELAVENT CHUNK
RETRIEVED CHUNKS
AI:Hello Kushal,

I apologize, but the provided information does not con